# Train a Gaussian splat on robot 1

Crab Team N. Run this on a rented CUDA GPU: Colab T4 or A100, RunPod, Lambda. It will not run on Apple
Silicon; `gsplat` needs a CUDA rasteriser and MPS is not one.

**What this does**

- Takes the COLMAP reconstruction already solved for robot 1. The poses are the expensive part and they exist.
- Forces **one shared fixed focal length** on every image before training.
- Trains 3D Gaussian splatting with `gsplat` and exports a `.ply` splat.

**What it does not do**

- It does not re-run SfM. Upload the solved model.
- It does not calibrate the camera. Nothing here can. See the focal section.
- It does not make a bad capture good. Read the last section before you judge the result.

| Fact | Value | Source |
| --- | --- | --- |
| Registered images | 70 of 125 | `docs/robot1-reconstruction.md` |
| Mean reprojection error | 0.790 px | COLMAP `model_analyzer` |
| Image size | 320 x 240 | ESP32-CAM QVGA |
| Drive | 2.17 m, straight, no turn | telemetry, two independent records |
| Focal length | **not identifiable** | fixed-focal sweep, 8 values |


## 1. Parameters

`SHARED_FOCAL_PX` is the only value you must decide. Everything else has a working default.


In [ ]:
# ---------------------------------------------------------------------------
# SHARED FIXED FOCAL LENGTH, in pixels, at 320 x 240.
#
# This is an ASSUMPTION, not a measurement. It is the single most load-bearing
# number in this notebook, for two reasons:
#
#   1. Focal length is NOT IDENTIFIABLE from this footage. A fixed-focal sweep
#      registered 70 of 70 images at every value from 120 to 550 px, and mean
#      reprojection error moved only 0.092 px across that 4.6x range. Nine
#      independent COLMAP runs on the same physical camera returned nine
#      different focals, spread 3.6x. Any single 'solved' focal from this
#      dataset is noise.
#
#   2. Two robots reconstructing one room CANNOT be merged with independent
#      intrinsics. Measured on this data: 64.8% scale error with independent
#      focals against 0.5% with one shared fixed focal. With independent
#      intrinsics the two models are related by a projective transform, not a
#      similarity, so no 7-DoF fit can align them.
#
# BRACKET: 160 to 300 px. Taken from the sweep, as the band within 0.02 px of
# the reprojection minimum. It is HFOV 56 to 90 deg. It rejects 120 px and
# everything at or above 340 px. It is a bracket, not a measurement.
#
# DEFAULT: 236 px. An OV2640 with the common 2.65 mm lens predicts about this,
# HFOV about 68 deg. That is a datasheet expectation. It sits inside the
# bracket, which is mild corroboration and nothing more.
#
# DO NOT use COLMAP's own solved 142.4 px. It falls outside the bracket and
# implies a 97 deg lens with almost no distortion, which is not physical.
#
# TO FIX THIS PROPERLY: calibrate the camera once. A checkerboard, 30 frames,
# or a printed AprilTag of known size. Then set the measured value here and
# reuse it for every robot and every session. Nothing in this dataset is a
# substitute for that calibration.
# ---------------------------------------------------------------------------
SHARED_FOCAL_PX = 236.0

FOCAL_BRACKET   = (160.0, 300.0)   # admissible band from the sweep
MAX_STEPS       = 30_000           # gsplat default; ~15 min on a T4 at this image count
DATA_DIR        = '/content/robot1'
RESULT_DIR      = '/content/results/robot1'

assert FOCAL_BRACKET[0] <= SHARED_FOCAL_PX <= FOCAL_BRACKET[1], (
    f'{SHARED_FOCAL_PX} px is outside the admissible bracket {FOCAL_BRACKET}. '
    'Change it, or delete this assert and record in your write-up that you did.')
print(f'shared fixed focal: {SHARED_FOCAL_PX} px  (bracket {FOCAL_BRACKET[0]}-{FOCAL_BRACKET[1]} px)')


## 2. Check the GPU

Stop here if this cell fails. There is no CPU path.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA device. gsplat cannot run. Rent a GPU runtime.'


## 3. Upload the COLMAP workspace

Upload **`robot1_splat_input.zip`**, built from `out/robot1/splat_input/` in the repo.

Expected layout after unzip, exactly this:

```
robot1/
  images/            70 PNG files, 000013.png .. 000082.png, 320x240
  sparse/0/
    cameras.bin      1 camera, currently SIMPLE_RADIAL f=142.4
    images.bin       70 poses
    points3D.bin     2341 sparse points
```

- `images/` holds only the **70 registered** frames. The other 55 frames of the session never registered:
  they are a stationary view of a blank wall and carry a median of 99 SIFT keypoints against 382 for the
  registered frames. Including them adds nothing and slows training.
- `sparse/0` is a copy of `out/robot1/work/ds/sparse/**1**`. COLMAP numbers sub-models by discovery order,
  not size: on this dataset `sparse/0` is a 3-image stub. Reading it succeeds silently and returns nothing.
- gsplat's parser looks for `sparse/0/` first, then `sparse/`. Keep the `0`.

To build the zip on the source machine:

```bash
cd out/robot1 && zip -qr robot1_splat_input.zip splat_input
```


In [ ]:
import os, zipfile, shutil, glob

try:
    from google.colab import files
    up = files.upload()                      # pick robot1_splat_input.zip
    zip_path = list(up)[0]
except ImportError:
    zip_path = 'robot1_splat_input.zip'      # RunPod / Lambda: put it beside this notebook

shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs('/content', exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/_unz')

# find the folder that actually contains images/ and sparse/, whatever it is named
root = None
for d, subs, _ in os.walk('/content/_unz'):
    if 'images' in subs and 'sparse' in subs:
        root = d; break
assert root, 'No folder containing both images/ and sparse/ was found in the zip.'
shutil.move(root, DATA_DIR)

imgs = sorted(glob.glob(f'{DATA_DIR}/images/*'))
sp   = f'{DATA_DIR}/sparse/0'
print(f'{len(imgs)} images, {imgs[0].split("/")[-1]} .. {imgs[-1].split("/")[-1]}')
for f in ('cameras.bin','images.bin','points3D.bin'):
    p = f'{sp}/{f}'
    assert os.path.exists(p), f'missing {p}'
    print(f'  {f:14s} {os.path.getsize(p):>8d} bytes')


## 4. Force one shared fixed focal

**This is the critical cell.** It rewrites `cameras.bin` to a single `PINHOLE` camera at `SHARED_FOCAL_PX`
and points every image at it.

- `gsplat` reads intrinsics straight from `cameras.bin` and **never refines them**. Whatever is in that file
  is the focal used for all 30,000 steps. Rewriting the file is therefore the whole mechanism.
- `PINHOLE` carries no distortion parameters, so gsplat skips its undistortion path. The original model was
  `SIMPLE_RADIAL` with `k1 = -0.072`, which is mild; dropping it is a deliberate, recorded simplification.
- Principal point is set to the image centre. The solved value was already the centre.
- Poses are **not** touched. The cell verifies that byte-for-byte.

Binary layouts, for reference:

```
cameras.bin  uint64 n, then per camera: int32 id, int32 model, uint64 w, uint64 h, float64 params[]
             model 0 SIMPLE_PINHOLE(f,cx,cy)   1 PINHOLE(fx,fy,cx,cy)   2 SIMPLE_RADIAL(f,cx,cy,k1)
images.bin   uint64 n, then per image: int32 id, float64 qvec[4], float64 tvec[3],  <- 60 bytes
             int32 camera_id, char name[] + NUL, uint64 m, then m x (float64 x, float64 y, int64 pt3D_id)
```


In [ ]:
import struct, shutil

CAM = f'{DATA_DIR}/sparse/0/cameras.bin'
IMG = f'{DATA_DIR}/sparse/0/images.bin'
NPARAMS = {0:3, 1:4, 2:4, 3:5, 4:8, 5:4, 6:12, 7:5, 8:4, 9:3, 10:4, 11:5}
MODEL   = {0:'SIMPLE_PINHOLE', 1:'PINHOLE', 2:'SIMPLE_RADIAL', 3:'RADIAL', 4:'OPENCV'}

def read_cameras(path):
    out = []
    with open(path, 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            cid, model, W, H = struct.unpack('<iiQQ', f.read(24))
            k = NPARAMS.get(model, 4)
            out.append((cid, model, W, H, list(struct.unpack(f'<{k}d', f.read(8*k)))))
    return out

def read_images(path):
    """Returns (image_id, camera_id, name, qvec, tvec, n_points2D) per image."""
    out = []
    with open(path, 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            iid = struct.unpack('<i', f.read(4))[0]
            q = struct.unpack('<4d', f.read(32)); t = struct.unpack('<3d', f.read(24))
            cid = struct.unpack('<i', f.read(4))[0]
            nm = b''
            while True:
                c = f.read(1)
                if c == b'\x00': break
                nm += c
            m = struct.unpack('<Q', f.read(8))[0]; f.read(m*24)
            out.append((iid, cid, nm.decode(), q, t, m))
        assert f.read() == b'', 'trailing bytes in images.bin: layout mismatch'
    return out

before_cam = read_cameras(CAM)
before_img = read_images(IMG)
cid, model, W, H, params = before_cam[0]
print(f'before: {len(before_cam)} camera(s), {MODEL.get(model, model)} {W}x{H} params={[round(p,4) for p in params]}')
print(f'        {len(before_img)} images, camera ids {sorted({i[1] for i in before_img})}')

shutil.copy(CAM, CAM + '.orig'); shutil.copy(IMG, IMG + '.orig')

# --- one PINHOLE camera, id 1 ---
with open(CAM, 'wb') as f:
    f.write(struct.pack('<Q', 1))
    f.write(struct.pack('<iiQQ', 1, 1, W, H))
    f.write(struct.pack('<4d', SHARED_FOCAL_PX, SHARED_FOCAL_PX, W/2.0, H/2.0))

# --- every image points at camera 1; the 60-byte pose block is copied untouched ---
raw = open(IMG + '.orig', 'rb').read()
out = bytearray(); i = 0
n = struct.unpack_from('<Q', raw, i)[0]; out += raw[i:i+8]; i += 8
for _ in range(n):
    pose = raw[i:i+60]; i += 60          # int32 id + 4 float64 qvec + 3 float64 tvec
    i += 4                               # old camera_id, discarded
    j = raw.index(b'\x00', i); name = raw[i:j+1]; i = j + 1
    m = struct.unpack_from('<Q', raw, i)[0]; i += 8
    pts = raw[i:i+m*24]; i += m*24
    out += pose + struct.pack('<i', 1) + name + struct.pack('<Q', m) + pts
assert i == len(raw), f'parsed {i} of {len(raw)} bytes: layout mismatch, do not use the result'
open(IMG, 'wb').write(bytes(out))

# --- verify ---
after_cam = read_cameras(CAM)
after_img = read_images(IMG)
acid, amodel, aW, aH, aparams = after_cam[0]
print(f'after : {len(after_cam)} camera(s), {MODEL.get(amodel, amodel)} {aW}x{aH} params={aparams}')
assert len(after_cam) == 1 and amodel == 1 and aparams[0] == aparams[1] == SHARED_FOCAL_PX
assert (aW, aH) == (W, H)
assert len(after_img) == len(before_img), 'image count changed'
assert {i[1] for i in after_img} == {1}, 'some image still points at another camera'
assert [(i[0], i[2], i[3], i[4], i[5]) for i in before_img] == \
       [(i[0], i[2], i[3], i[4], i[5]) for i in after_img], 'a pose or name changed: abort'
print(f'        {len(after_img)} images, all on camera 1; ids, names, poses and observations unchanged')
print(f'        every image now trains at fx = fy = {SHARED_FOCAL_PX} px')


## 5. Install gsplat

- Pinned to **gsplat 1.5.3** and the matching repo tag `v1.5.3`, so the trainer and the library cannot drift
  apart. Both verified to exist: PyPI serves `gsplat-1.5.3-py3-none-any.whl`, GitHub serves the tag.
- That wheel is **pure Python**. It carries no compiled CUDA. gsplat JIT-compiles its kernels on the first
  `import gsplat`, through `torch.utils.cpp_extension`. So `pip install` returns in seconds and the real
  wait, roughly 3 to 8 minutes, happens at first import. It is cached afterwards, per runtime.
- The trainer lives in the repo, not the package, so the repo is cloned at the same tag.
- `examples/requirements.txt` pins its own git dependencies by commit (`pycolmap`, `nerfview`, `fused-ssim`,
  `fused-bilagrid`). Use that file rather than re-pinning them by hand; `fused-ssim` is not on PyPI at all.
- That file also pins **`numpy<2.0.0`**. If the runtime ships NumPy 2.x, pip downgrades it and you must
  **restart the runtime once**, then re-run sections 1 and 5. The check cell below refuses to continue
  otherwise.
- Pre-built CUDA wheels exist at `docs.gsplat.studio/whl/<ptXXcuYYY>` but only for cp310 and torch 2.0-2.4.
  Current Colab matches none of them, which is why this notebook takes the JIT route instead.


In [ ]:
GSPLAT_VERSION = '1.5.3'

import subprocess, sys, time, torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)

for args in ([ 'ninja', 'jaxtyping', 'rich' ],
             [ f'gsplat=={GSPLAT_VERSION}' ]):
    rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
    assert rc == 0, f'pip install {args} failed'

subprocess.call(['rm', '-rf', '/content/gsplat_src'])
rc = subprocess.call(['git', 'clone', '-q', '--depth', '1', '--branch', f'v{GSPLAT_VERSION}',
                      'https://github.com/nerfstudio-project/gsplat.git', '/content/gsplat_src'])
assert rc == 0, 'clone of the gsplat repo failed'
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                      '/content/gsplat_src/examples/requirements.txt'])
assert rc == 0, 'examples/requirements.txt failed to install'
print('installed')


In [ ]:
import numpy
print('numpy', numpy.__version__)
assert numpy.__version__.startswith('1.'), (
    'NumPy is still 2.x. gsplat examples require <2.0.0. Restart the runtime now '
    '(Runtime > Restart session), then re-run sections 1 and 5. Do not skip this.')

# First import compiles the CUDA kernels. This is the slow step, not pip.
import time
t0 = time.time()
from gsplat.cuda._backend import _C          # forces the JIT build now, not mid-training
import gsplat
print(f'gsplat {gsplat.__version__} ready in {time.time()-t0:.0f} s')
from gsplat import export_splats
print('export_splats available: .ply export will work')


## 6. Train

- `--data_factor 1` uses `images/` as-is. Any other value makes the parser look for `images_2/`, `images_4/`.
  There is nothing to downscale at 320 x 240.
- `--disable_viewer` is required: the viser viewer opens a port that Colab will not serve.
- `--save_ply` with `--ply_steps` is what produces the splat file. It is **off by default** in this trainer.
- The intrinsics written in section 4 are used verbatim. `simple_trainer` refines Gaussians and, optionally,
  poses. It never refines focal length, which is the point.
- 70 images at 320 x 240 is a very small job. Expect roughly 10 to 20 minutes on a T4, less on an A100.


In [ ]:
import os, subprocess, sys
os.makedirs(RESULT_DIR, exist_ok=True)

# Built as a list and run through subprocess rather than a multi-line `!` line:
# backslash continuation inside an IPython shell escape is not dependable.
cmd = [sys.executable, 'simple_trainer.py', 'default',
       '--data_dir', DATA_DIR,
       '--data_factor', '1',
       '--result_dir', RESULT_DIR,
       '--max_steps', str(MAX_STEPS),
       '--save_ply',
       '--ply_steps', str(MAX_STEPS),
       '--disable_viewer']
print(' '.join(cmd))
rc = subprocess.call(cmd, cwd='/content/gsplat_src/examples')
assert rc == 0, f'simple_trainer exited {rc}. Read the traceback above; do not trust any file it left behind.'


## 7. Export and download the splat


In [ ]:
import glob, os
plys = sorted(glob.glob(f'{RESULT_DIR}/ply/*.ply'), key=os.path.getmtime)
assert plys, (f'No .ply under {RESULT_DIR}/ply. Training did not reach a ply step. '
              'Check the trainer output above for the real error.')
ply = plys[-1]
print(f'{ply}  {os.path.getsize(ply)/1e6:.1f} MB')

# splat count, read from the PLY header rather than assumed
with open(ply, 'rb') as f:
    head = f.read(2048).split(b'end_header')[0].decode('ascii', 'replace')
n = [l for l in head.splitlines() if l.startswith('element vertex')][0].split()[-1]
print(f'{int(n):,} gaussians')

try:
    from google.colab import files
    files.download(ply)
except ImportError:
    print('Not on Colab: copy the file off the machine yourself.')


## 8. View the splat

The exported file is a standard 3DGS `.ply`: position, scale, rotation quaternion, opacity, and spherical
harmonic colour per Gaussian. Any of these opens it.

| Viewer | Where | Note |
| --- | --- | --- |
| PlayCanvas SuperSplat | `superspl.at/editor` | Browser, no install. Also edits and crops. |
| antimatter15 splat | `antimatter15.com/splat/` | Browser, minimal, very fast. |
| `gsplat` viewer | `python simple_viewer.py --ckpt <ckpt>.pt` | Local, needs the CUDA machine. |
| Blender | 3DGS Render add-on | If you need to composite. |

- Load the **`.ply` from `results/robot1/ply/`**, not the `.pt` checkpoint. The `.pt` is for resuming.
- The scene is about 1 m across in COLMAP units. Most viewers auto-frame; if not, zoom in hard.


## 9. What to expect, honestly

This capture is close to the worst case for 3DGS. Expect a poor splat, and expect it for reasons that are
already measured rather than guessed.

**Against it**

- **320 x 240, heavily compressed.** Source JPEGs are 3 to 6 KB. Median 116 SIFT keypoints per frame; an
  earlier phone clip of the same kind of scene gave 1185. There is very little high-frequency detail for
  Gaussians to latch onto.
- **Forward motion.** The robot drove straight down the optical axis and never turned. The epipole falls
  *inside* the 320 x 240 frame for about half the moving pairs. Points straight ahead barely shift between
  frames, so their depth is poorly conditioned, and 3DGS inherits that.
- **Thin baseline.** Median consecutive baseline over median scene depth is 0.063, just above the level
  where triangulation is stable.
- **No rotation.** 0.1 s of steering in 49.3 s. Rotation is what makes structure observable.
- **Only 70 views, all from one line.** Every Gaussian is supervised from one direction. Anything the
  camera never passed will be unconstrained, and 3DGS fills unconstrained space with confident nonsense.
- **Focal is assumed.** A wrong focal does not stop training; it bends the scene. There is no residual in
  this dataset that will tell you it is wrong.

**Likely appearance**

- Recognisable near the drive line, smeared or spiky to the sides, and hollow behind anything.
- Good training PSNR on held-out frames from the same straight line does **not** mean good geometry. Those
  frames share the viewpoint. Orbit the result; that is the real test.

**What to capture instead**

From `docs/video-capture-guide.md`, which was written after the first capture failed:

- **Walk sideways past the scene. Do not stand still and sweep.** Point the camera at one region and strafe
  across it, keeping that region in frame. Strafing costs the same effort as walking forward and gives depth
  everywhere; forward motion gives depth only at the sides.
- Travel 1 to 2 m while the same wall or object stays in view. Move at about a third of walking pace.
- **Turn.** Rotation combined with translation is what makes structure observable.
- Sample near **2.5 fps**, not 12. Measured on this session: 276 AVI frames at 12 fps registered 276 of 276
  and still produced a path 38% too long, against 2% for 70 frames at 2.5 fps. Registration rate is not a
  quality metric.
- **Do not idle with the camera running.** 60% of this session, and 55 of 125 frames, were a stationary
  view of a blank wall.
- **Put a printed AprilTag of known size in view** at the start and end of every run. It fixes all 7 degrees
  of freedom at once, including the roll about the drive axis that a straight-line camera path can never
  observe. See `docs/tag-size-rule.md`.
- **Calibrate the camera once**, then hard-code that focal here and in every robot.


## 10. Fallback: nerfstudio splatfacto

If `gsplat`'s example trainer will not build, `nerfstudio` wraps the same rasteriser with a heavier but
better-trodden install. Splatfacto does not need tiny-cuda-nn.

```bash
pip install nerfstudio==1.1.5
ns-train splatfacto --data /content/robot1 --max-num-iterations 30000 \
    --viewer.quit-on-train-completion True colmap --downscale-factor 1
ns-export gaussian-splat --load-config outputs/.../config.yml --output-dir exports/splat
```

**Run section 4 first either way.** Nerfstudio's COLMAP dataparser also reads intrinsics straight from
`cameras.bin`, so the shared fixed focal must already be written into the file. Skipping that step is the
one mistake that quietly ruins the result and cannot be detected from the training loss.
